# تحليل رسم الاستدعاءات واكتشاف المكونات شديدة الترابط (SCC)

يستكشف هذا الدفتر من دفاتر الملاحظات ميزات التحليل البرمجي المتقدمة في UnifyWeaver:

- **بناء رسم الاستدعاءات (Call Graph Construction)** - بناء مخططات الاعتماديات من كود Prolog
- **اكتشاف SCC** - العثور على المكونات شديدة الترابط (العودية المتبادلة)
- **تحليل الأنماط** - فهم أنماط العودية
- **العرض المرئي للاعتماديات** - تمثيل العلاقات بين المحددات بصريًا

## الأهداف التعليمية

- فهم كيف يحلل UnifyWeaver بنية الكود
- بناء وفحص رسوم الاستدعاءات
- اكتشاف العودية المتبادلة باستخدام خوارزمية تارجان (Tarjan)
- التمثيل المرئي لاعتماديات الكود

## الإعداد

تحميل UnifyWeaver ووحدات التحليل.

In [ ]:
% تحميل التهيئة
['../init'].

% تحميل وحدات التحليل
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## المثال 1: رسم استدعاءات بسيط

دعنا نبدأ بمحدد بسيط ونبني رسم الاستدعاءات الخاص به.

In [ ]:
% تعريف محدد ancestor
:- dynamic ancestor/2.
:- dynamic parent/2.

% حقائق parent
parent(abraham, isaac).
parent(isaac, jacob).

% قواعد ancestor
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### بناء رسم الاستدعاءات

In [ ]:
% بناء رسم الاستدعاءات لـ ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### تحليل الاعتماديات

In [ ]:
% الحصول على جميع اعتماديات ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% التحقق مما إذا كان عوديًا ذاتيًا
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## المثال 2: اكتشاف العودية المتبادلة

دعنا الآن نكتشف العودية المتبادلة باستخدام مثال الزوجي/الفردي.

In [ ]:
% تعريف محددات ذات عودية متبادلة
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### بناء رسم الاستدعاءات لكلا المحـددين

In [ ]:
% بناء رسم الاستدعاءات لكلا المحددین
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### العثور على المكونات شديدة الترابط (SCCs)

In [ ]:
% إعادة بناء الرسم لأن المتغيرات لا تستمر بين خلايا دفتر الملاحظات
build_call_graph([is_even/1, is_odd/1], _Graph),
% العثور على SCCs باستخدام خوارزمية تارجان
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### التحقق مما إذا كان SCC بسيطًا (Trivial)

In [ ]:
% إعادة بناء القيم المشتقة حتى تعمل هذه الخلية أيضًا بشكل مستقل
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% فحص كل SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## المثال 3: رسم استدعاءات معقد

دعنا نحلل نظامًا أكثر تعقيدًا يحتوي على محددات متعددة.

In [ ]:
% تعريف برنامج صغير يحتوي على محددات متعددة
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent يستخدم parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: نفس الوالد، أطفال مختلفون
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: الآباء إخوة
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### بناء رسم الاستدعاءات الكامل

In [ ]:
% بناء رسم الاستدعاءات لجميع المحددات
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### العثور على مجموعات المحددات

العثور على مجموعة المحددات ذات العودية المتبادلة التي تحتوي على محدد البداية.

In [ ]:
% العثور على مجموعة العودية المتبادلة التي تحتوي على cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## المثال 4: اكتشاف الأنماط

دعنا نستخدم مطابقات الأنماط لتحليل أنواع العودية.

In [ ]:
% تعريف أنماط عودية متنوعة
:- dynamic count/3.     % عودية الذيل
:- dynamic factorial/2. % عودية خطية
:- dynamic fib/2.       % عودية شجرية (أو خطية إذا تم اكتشافها)

% العد بعودية الذيل
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% المضروب بالعودية الخطية
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% فيبوناتشي (يمكن اكتشافها كعودية خطية أو شجرية)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### اكتشاف عودية الذيل

In [ ]:
% التحقق مما إذا كان count/3 يعتمد عودية الذيل
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### اكتشاف العودية الخطية

In [ ]:
% التحقق مما إذا كان factorial/2 يعتمد عودية خطية
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### عد الاستدعاءات العودية

In [ ]:
% عد الاستدعاءات العودية في فيبوناتشي
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## العرض المرئي بصيغة DOT

دعنا ننشئ تمثيل Graphviz DOT لرسم الاستدعاءات الخاص بنا.

In [ ]:
% دالة مساعدة لإنشاء صيغة DOT
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% إنشاء DOT لرسم الزوجي/الفردي
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### حفظ ملف DOT

In [ ]:
% إعادة بناء مصدر DOT لأن المتغيرات لا تستمر بين الخلايا
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## تمرين: حلل الكود الخاص بك

جرب تعريف محدداتك الخاصة وتحليلها!

In [ ]:
% حدد محدداتك هنا
% ثم قم ببناء رسوم الاستدعاءات، والعثور على SCCs، واكتشاف الأنماط

% مثال:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## الملخص

في هذا الدفتر، تعلمت:

✅ كيفية بناء رسوم الاستدعاءات من كود Prolog

✅ كيفية اكتشاف المكونات شديدة الترابط (SCCs) للعودية المتبادلة

✅ كيفية استخدام مطابقات الأنماط لتصنيف أنواع العودية

✅ كيفية تحليل اعتماديات المحددات

✅ كيفية عرض رسوم الاستدعاءات مرئيًا بصيغة DOT

## موضوعات متقدمة

لمزيد من التحليل المتقدم:

- **الترتيب الطوبولوجي (Topological Ordering)**: استخدم `topological_order/2` لترتيب المكونات شديدة الترابط (SCCs) حسب الاعتماديات
- **مطابقات الأنماط المخصصة**: كتابة محددات الكشف عن الأنماط الخاصة بك
- **استخراج نمط المجمع**: استخدم `extract_accumulator_pattern/2` للتحليل التفصيلي
- **حظر العودية الخطية**: استخدم `forbid_linear_recursion/1` لفرض استراتيجيات تجميع مختلفة

## المراجع والملفات ذات الصلة

- الفصل 10: استبطان Prolog ونظريته
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`